# Amazon ML Challenge 2026 (Data Inspection Notebook)

In [1]:
import pandas as pd 
import numpy as np 

## Ground Truth Dataset 

In [2]:
data=pd.read_csv('dataset/train/train_ground_truth.tsv',sep="\t")

In [3]:
print("Ground Truth Dataset Shape:", data.shape)

Ground Truth Dataset Shape: (2206821, 2)


In [4]:
data.head(5)

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


## Source 1 Data Inspection

In [5]:
source1=pd.read_csv('dataset/train/train_source1.tsv',sep="\t")

In [6]:
print("Source 1 Shape:", source1.shape)

Source 1 Shape: (2206821, 4)


In [7]:
print(source1.head(5))

      entity_id            business_name  \
0  S1-925783039      Orelee's Barbershop   
1  S1-773889195              Prime Money   
2  S1-377745466            B+ Retail Inc   
3  S1-133037285            Christ Chapel   
4  S1-755362802  Prabhav Business Center   

                                    business_address country  
0             1795 Westchester Drive, High Point, NC      US  
1                    17560 Ellis Road, Tahlequah, OK      US  
2                1712 Montebello Avenue, Phoenix, AZ      US  
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD      US  
4  797, Lake Town Block A, Kolkata, Howrah, West ...   India  


## Source 2 Data Inspection

In [8]:
source2=pd.read_csv('dataset/train/train_source2.tsv',sep="\t")

In [9]:
print("Source 2 Shape:", source2.shape)

Source 2 Shape: (5034616, 4)


In [10]:
print(source2.head(5))

      entity_id                    business_name  \
0  S2-166376419  राम मार्केटिंग प्राइवेट लिमिटेड   
1  S2-764573417     -- Holloway Peak Inc Seafood   
2  S2-639257739         आदित्य प्रॉपर्टीज एलएलपी   
3  S2-163963287                       Summit Inc   
4   S2-49942811     Delta Tetlecommunication Inc   

                                   business_address country  
0      KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi   India  
1                         105 ELM ST, MORGANTON, NC      US  
2  G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh   India  
3             GREENSBORO, NC, 19 1/2 STARDUST TRAIL      US  
4                   914 PIERPONT AVE, CLEVELAND, OH      US  


## Source 3 Data Inspection

In [11]:
source3=pd.read_csv('dataset/train/train_source3.tsv',sep="\t")

In [12]:
print("Source 3 Shape:", source3.shape)

Source 3 Shape: (5285603, 4)


In [13]:
print(source3.head(5))

      entity_id                                business_name  \
0  S3-202863386                           wilfordhancock.com   
1  S3-859268022  International South Consultants Private Ltd   
2   S3-22467283                  LLC Moncada Léarning Center   
3  S3-671162755                               Moyna's Coffee   
4  S3-960981775                 Pvt. EFS Print Ventures Ltd.   

                                    business_address country  
0                        Mack Rd, Haltom City, Texas      US  
1                                                NaN   India  
2                    5780 Fawn Ct, Fort Worth, Texas      US  
3       1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio      US  
4  Door No 183, 41St Cross, 22Nd Main 9Th Block J...   India  


## Dataset Schema Inspection

In [14]:
print(source1.columns.tolist())
print(source2.columns.tolist())
print(source3.columns.tolist())
print(data.columns.tolist())

['entity_id', 'business_name', 'business_address', 'country']
['entity_id', 'business_name', 'business_address', 'country']
['entity_id', 'business_name', 'business_address', 'country']
['source1_entity_id', 'matched_entity_ids']


## Preparing Ground Truth for Training

### For Single Row

In [15]:
row = data.iloc[0] # Selecting 1 row from the ground truth dataset 
print(row) # Getting the S1 ID and Matching ones from S2 and S3

source1_entity_id                                             S1-965667
matched_entity_ids    S2-681193310,S2-743505751,S3-775321672,S3-1129...
Name: 0, dtype: object


In [16]:
s1_id = row["source1_entity_id"] # Selecting the S1 ID 
print(s1_id)

S1-965667


In [17]:
s1_record = source1[source1["entity_id"] == s1_id] # Entire Record (Row) for the S1 ID 
print(s1_record)

         entity_id                 business_name  \
1286323  S1-965667  Maure Williams Colombier Inc   

                         business_address country  
1286323  85 Wayne Avenue, Ticonderoga, NY      US  


In [18]:
matched_ids = row["matched_entity_ids"].split(",") # All the matching IDs in S2 and S3 seperated by a comma 
print(matched_ids)

['S2-681193310', 'S2-743505751', 'S3-775321672', 'S3-11291185', 'S3-860443364']


In [19]:
pairs=[] # Data Frame

In [21]:
for target_id in matched_ids: # For all S2 and S3 IDs
    target_id = target_id.strip() # Removing the extra space 

    if target_id.startswith("S2"): # Checking base source S2/S3
        target_record = source2[source2["entity_id"] == target_id].iloc[0] # Record ID for further extraction 
        target_source = "Source 2" # Source Extracted 

    # Similar for S3 
    elif target_id.startswith("S3"):
        target_record = source3[source3["entity_id"] == target_id].iloc[0]
        target_source = "Source 3"

    # Appending in the List 
    
    pairs.append({
        "S1_ID": s1_record["entity_id"], # Base Entity from S1 
        "S1_Name": s1_record["business_name"], 
        "S1_Address": s1_record["business_address"], 
        "S1_Country": s1_record["country"],

        # Target Entity Extracted from S1/S2 
        "Target_ID": target_record["entity_id"],
        "Target_Name": target_record["business_name"],
        "Target_Address": target_record["business_address"],
        "Target_Country": target_record["country"],

        # Target Source 
        "Target_Source": target_source
    })

In [22]:
pairs_df = pd.DataFrame(pairs) # Converting to Data frame
pairs_df

,S1_ID,S1_Name,S1_Address,S1_Country,Target_ID,Target_Name,Target_Address,Target_Country,Target_Source
0,"1286323 S1-965667 Name: entity_id, dtype: o...",1286323 Maure Williams Colombier Inc Name: ...,"1286323 85 Wayne Avenue, Ticonderoga, NY Na...","1286323 US Name: country, dtype: object",S2-681193310,Maure Wilblims Colombier Inc,NaN,US,Source 2
1,"1286323 S1-965667 Name: entity_id, dtype: o...",1286323 Maure Williams Colombier Inc Name: ...,"1286323 85 Wayne Avenue, Ticonderoga, NY Na...","1286323 US Name: country, dtype: object",S2-743505751,Maure Williams Colombier,NaN,US,Source 2
2,"1286323 S1-965667 Name: entity_id, dtype: o...",1286323 Maure Williams Colombier Inc Name: ...,"1286323 85 Wayne Avenue, Ticonderoga, NY Na...","1286323 US Name: country, dtype: object",S3-775321672,Dréxkor,"85 Wanye Avenue, Ticonderoga Townshiip, New York",US,Source 3
3,"1286323 S1-965667 Name: entity_id, dtype: o...",1286323 Maure Williams Colombier Inc Name: ...,"1286323 85 Wayne Avenue, Ticonderoga, NY Na...","1286323 US Name: country, dtype: object",S3-11291185,maurewilliamscolombier.com,"Wayne Ave, Ticonderoga Townshiip, New York",US,Source 3
4,"1286323 S1-965667 Name: entity_id, dtype: o...",1286323 Maure Williams Colombier Inc Name: ...,"1286323 85 Wayne Avenue, Ticonderoga, NY Na...","1286323 US Name: country, dtype: object",S3-860443364,Maure Williams Inc Center,NaN,US,Source 3


In [29]:
match_counts = (
   data["matched_entity_ids"]
    .fillna("")
    .str.count(",") + 1
)

match_counts = match_counts.where(
    data["matched_entity_ids"].notna(),
    0
)

print(match_counts.describe())

count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: matched_entity_ids, dtype: float64


In [30]:
match_counts.value_counts().sort_index()

matched_entity_ids
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64